机器学习的目标是发现模式，即学习样本集潜在的总体规律，从而做到泛化

**独立同分布假设：**
- 所有数据都遵循同样的分布
- 对数据进行独立抽取，即抽样的过程中不存在“记忆”

影响模型泛化能力的因素：
- 可调整参数的数量
- 参数的取值
- 训练样本的数量

当训练数据很少时，可以采用**K折交叉验证**

In [ ]:
import torch
from torch import nn
import d2l
import warnings
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams["font.family"] = "Microsoft YaHei"
plt.rcParams["axes.unicode_minus"] = False

# 损失函数 evaluate_loss
def evaluate_loss(net, data_iter, loss):
    """评估给定数据集上模型的平均损失"""
    metric = d2l.Accumulator(2)  # 累加总损失、总样本数
    for X, y in data_iter:
        out = net(X)
        y = y.reshape(out.shape)
        l = loss(out, y)
        metric.add(l.sum(), l.numel())
    return metric[0] / metric[1]

# 训练函数
def train(train_features, test_features, train_labels, test_labels, num_epochs=400):
    # 损失函数：均方误差，返回每个样本单独损失
    loss = nn.MSELoss(reduction='none')
    # 获取多项式特征维度（列数）
    input_shape = train_features.shape[-1]
    # 线性层不设偏置：多项式0次项已经包含常数1，无需额外bias
    net = nn.Sequential(nn.Linear(input_shape, 1, bias=False))
    # 批次大小，最少1条
    batch_size = min(10, train_labels.shape[0])

    # 构造训练、测试迭代器
    train_iter = d2l.load_array((train_features, train_labels.reshape(-1, 1)), batch_size)
    test_iter = d2l.load_array((test_features, test_labels.reshape(-1, 1)), batch_size, is_train=False)

    # SGD优化器
    trainer = torch.optim.SGD(net.parameters(), lr=0.01)

    # 动态绘图器：log坐标观察loss变化
    animator = d2l.Animator(
        xlabel='epoch', ylabel='loss', yscale='log',
        xlim=[1, num_epochs], ylim=[1e-3, 1e2],
        legend=['train', 'test']
    )

    # 多轮训练主循环
    for epoch in range(num_epochs):
        # 执行一轮完整训练（d2l内置单epoch训练逻辑，不用手搓梯度）
        d2l.train_epoch_ch3(net, train_iter, loss, trainer)
        # 每20轮记录一次损失绘图
        if epoch == 0 or (epoch + 1) % 20 == 0:
            train_l = evaluate_loss(net, train_iter, loss)
            test_l = evaluate_loss(net, test_iter, loss)
            animator.add(epoch + 1, (train_l, test_l))
    
    # 训练结束打印最终权重（多项式各项系数）
    print('weight:', net[0].weight.data.numpy())
    return net